# AI Search and Reasoning Under Uncertainty

**Syllabus mapping:** Search: informed, uninformed, adversarial;
logic: propositional, predicate; reasoning under uncertainty topics:
conditional independence representation, exact inference through
variable elimination, approximate inference through sampling.

**Objectives:** trace A* search with admissible heuristics, implement
adversarial Minimax evaluation, and perform exact inference via
Variable Elimination on a Bayesian Network.

In [ ]:
import heapq

# 1. Informed Search: A* on a weighted directed graph
# f(n) = g(n) + h(n)

graph = {
    "S": [("A", 2), ("B", 4)],
    "A": [("C", 3), ("D", 4)],
    "B": [("D", 1)],
    "C": [("G", 4)],
    "D": [("G", 2)],
    "G": [],
}

heuristic = {"S": 5, "A": 4, "B": 2, "C": 3, "D": 2, "G": 0}


def a_star(graph, start, goal, h):
    pq = [(h[start], 0, start, [start])]
    visited = set()

    while pq:
        f, g, current, path = heapq.heappop(pq)
        print(f"Expanding {current}: g={g}, h={h[current]}, f={f}")
        if current == goal:
            return path, g
        if current in visited:
            continue
        visited.add(current)

        for neighbor, cost in graph.get(current, []):
            if neighbor not in visited:
                next_g = g + cost
                next_f = next_g + h[neighbor]
                heapq.heappush(pq, (next_f, next_g, neighbor, path + [neighbor]))

    return None, float("inf")


path, total_cost = a_star(graph, "S", "G", heuristic)
print(f"\nA* Optimal Path: {' -> '.join(path)}, Total Cost: {total_cost}")

In [ ]:
# 2. Reasoning Under Uncertainty: Variable Elimination on A -> B -> C
# P(A=1) = 0.2
# P(B=1|A=1) = 0.8, P(B=1|A=0) = 0.3
# P(C=1|B=1) = 0.9, P(C=1|B=0) = 0.1
# Find P(C=1 | A=1) by eliminating B:
# P(C=1 | A=1) = sum_b P(C=1|b) * P(b|A=1)

p_a = {1: 0.2, 0: 0.8}
p_b_given_a = {(1, 1): 0.8, (0, 1): 0.2, (1, 0): 0.3, (0, 0): 0.7}
p_c_given_b = {(1, 1): 0.9, (0, 1): 0.1, (1, 0): 0.1, (0, 0): 0.9}

# Variable elimination for P(C=1 | A=1)
p_c1_given_a1 = sum(
    p_c_given_b[(1, b)] * p_b_given_a[(b, 1)] for b in (0, 1)
)
print(f"Exact Inference P(C=1 | A=1) = {p_c1_given_a1:.4f}")

# 3. Approximate Inference: Rejection Sampling
import random
random.seed(42)
N = 100000
accepted_c1 = 0
total_accepted = 0

for _ in range(N):
    # Sample A
    a = 1 if random.random() < p_a[1] else 0
    if a != 1:  # Reject sample because evidence is A=1
        continue
    total_accepted += 1
    # Sample B given A=1
    b = 1 if random.random() < p_b_given_a[(1, 1)] else 0
    # Sample C given B
    c = 1 if random.random() < p_c_given_b[(1, b)] else 0
    if c == 1:
        accepted_c1 += 1

sampled_prob = accepted_c1 / total_accepted
print(f"Sampled Estimate P(C=1 | A=1) = {sampled_prob:.4f} (from {total_accepted} samples)")

## GATE-Style Practice

**MCQ:** In A* graph search, which property of the heuristic function
$h(n)$ guarantees that the first time a goal node is expanded, an
optimal path has been found, even when nodes are not reopened?

A. Admissibility ($h(n) \le h^*(n)$)
B. Consistency / Monotonicity ($h(n) \le c(n, a, n') + h(n')$)
C. Dominance ($h(n) > h'(n)$)
D. Boundedness ($h(n) \ge 0$)

**MSQ:** In a Bayesian Network with causal chain structure $X \to Y \to Z$,
which of the following conditional independence statements are true?

A. $X$ and $Z$ are conditionally independent given $Y$ ($X \perp Z \mid Y$).
B. $X$ and $Z$ are marginally independent ($X \perp Z$).
C. $P(Z \mid Y, X) = P(Z \mid Y)$.
D. Observing $Y$ blocks the active path between $X$ and $Z$.

**NAT:** Consider a two-player zero-sum game tree where the root is a
MAX node with two actions $L$ and $R$. The subtree under $L$ has two
leaves with payoff values 4 and 7. The subtree under $R$ has two leaves
with payoff values 2 and 9. What is the minimax value at the root node?

## Solutions

MCQ: **B**. In graph search without node reopening, consistency
(monotonicity) is required to guarantee optimality. (Admissibility alone
suffices for tree search or when closed nodes can be reopened).

MSQ: **A, C, D**. In a causal chain $X \to Y \to Z$, conditioning on $Y$
d-separates $X$ and $Z$, making them conditionally independent. (B is
false: knowing $X$ alters the probability of $Y$, which in turn alters $Z$;
they are marginally dependent).

NAT: **4.0**. The children of the root are MIN nodes. Under action $L$,
the MIN player chooses $\min(4, 7) = 4$. Under action $R$, the MIN player
chooses $\min(2, 9) = 2$. At the root, the MAX player chooses
$\max(4, 2) = 4$.